In [1]:
import math
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.style as style
import numpy as np
import pandas as pd

import seaborn as sns
from scipy import stats
from scipy.stats import chi2_contingency, f_oneway
import datetime
from sklearn.model_selection import train_test_split, GridSearchCV, RandomizedSearchCV, cross_val_score
from sklearn.preprocessing import StandardScaler, RobustScaler
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from sklearn.ensemble import VotingRegressor
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor

In [2]:
# 데이터 시각화
import matplotlib.pyplot as plt
import matplotlib

# 맑은 고딕 적용
matplotlib.rc("font", family = "AppleGothic")
# 음수 표시
matplotlib.rc("axes", unicode_minus = False)

In [3]:
df_users = pd.read_csv("df_users.csv", encoding="cp949")
df_users.head()

,idUser,Gender,Age,FamilyCount,MemberYN,AgeGroup
0,U10001,여성,26,2,Y,20
1,U10002,남성,61,2,Y,60
2,U10003,여성,34,2,Y,30
3,U10004,남성,26,1,N,20
4,U10005,여성,33,3,Y,30


In [4]:
df_users.isnull().sum()

idUser         0
Gender         0
Age            0
FamilyCount    0
MemberYN       0
AgeGroup       0
dtype: int64

In [5]:
df_items = pd.read_csv("clean_item.csv")
df_items = df_items.drop("Unnamed: 0",axis=1)
df_items.head()

,ItemLargeCode,ItemLargeName,ItemMiddleCode,ItemMiddleName,ItemSmallCode,ItemSmallName,ItemCode,ItemName,PriceYear,PriceMin,PriceMax
0,L1,가공식품,M11,곡물,S0080,국수,L1-M11-S0080-1001,(식품)샘표 김치국수 101g 10입,2023,15840,16130
1,L1,가공식품,M11,곡물,S0080,국수,L1-M11-S0080-1001,(식품)샘표 김치국수 101g 10입,2024,17030,17340
2,L1,가공식품,M11,곡물,S0080,국수,L1-M11-S0080-1001,(식품)샘표 김치국수 101g 10입,2025,17380,18640
3,L1,가공식품,M11,곡물,S0080,국수,L1-M11-S0080-1002,2.1kg 6배 메밀 Bestco 희석용 소바 국수장국,2024,14160,15350
4,L1,가공식품,M11,곡물,S0080,국수,L1-M11-S0080-1002,2.1kg 6배 메밀 Bestco 희석용 소바 국수장국,2025,15060,16160


In [6]:
df_items.isnull().sum()

ItemLargeCode     0
ItemLargeName     0
ItemMiddleCode    0
ItemMiddleName    0
ItemSmallCode     0
ItemSmallName     0
ItemCode          0
ItemName          0
PriceYear         0
PriceMin          0
PriceMax          0
dtype: int64

In [7]:
df_items.shape

(10051, 11)

In [8]:
df_orders = pd.read_csv("orders.csv", encoding="cp949")
df_orders.head()

,idUser,idOrder,OrderDT,ItemCode,Price,DeliveryDT,OrderYear,OrderMonth,OrderDay,OrderHour,OrderMinute,OrderWeekday,DeliveryYear,DeliveryMonth,DeliveryDay,DeliveryHour,DeliveryMinute,DeliveryWeekday,DeliveryDeadline,IsLate
0,U10001,U10001-O2023-1002,2023-01-06 17:08:51,L4-M17-S0530-1024,33310,2023-01-07 06:24:00,2023,1,6,17,8,Friday,2023,1,7,6,24,Saturday,2023-01-07 07:00:00,False
1,U10001,U10001-O2023-1002,2023-01-06 17:08:51,L1-M21-S0540-1082,3780,2023-01-07 06:24:00,2023,1,6,17,8,Friday,2023,1,7,6,24,Saturday,2023-01-07 07:00:00,False
2,U10001,U10001-O2023-1002,2023-01-06 17:08:51,L1-M15-S0140-1311,22520,2023-01-07 06:24:00,2023,1,6,17,8,Friday,2023,1,7,6,24,Saturday,2023-01-07 07:00:00,False
3,U10001,U10001-O2023-1002,2023-01-06 17:08:51,L4-M12-S0350-1035,21630,2023-01-07 06:24:00,2023,1,6,17,8,Friday,2023,1,7,6,24,Saturday,2023-01-07 07:00:00,False
4,U10001,U10001-O2023-1003,2023-01-13 16:50:14,L4-M12-S0640-1057,11700,2023-01-14 06:28:00,2023,1,13,16,50,Friday,2023,1,14,6,28,Saturday,2023-01-14 07:00:00,False


In [9]:
df_orders.isnull().sum()

idUser              0
idOrder             0
OrderDT             0
ItemCode            0
Price               0
DeliveryDT          0
OrderYear           0
OrderMonth          0
OrderDay            0
OrderHour           0
OrderMinute         0
OrderWeekday        0
DeliveryYear        0
DeliveryMonth       0
DeliveryDay         0
DeliveryHour        0
DeliveryMinute      0
DeliveryWeekday     0
DeliveryDeadline    0
IsLate              0
dtype: int64

# 합병 

In [10]:
# === 세 데이터셋 병합 ===

# 1단계: orders + items (ItemCode, 연도 기준)
df_merged = df_orders.merge(
    df_items,
    left_on=['ItemCode', 'OrderYear'],
    right_on=['ItemCode', 'PriceYear'],
    how='left'
)

# 2단계: + users (idUser 기준)
df_merged = df_merged.merge(
    df_users,
    on='idUser',
    how='left'
)

# PriceYear 중복 컬럼 제거 (OrderYear와 동일)
df_merged = df_merged.drop(columns=['PriceYear'])

print(f"병합 결과: {df_merged.shape}")
df_merged.head()

병합 결과: (854538, 34)


,idUser,idOrder,OrderDT,ItemCode,Price,DeliveryDT,OrderYear,OrderMonth,OrderDay,OrderHour,...,ItemSmallCode,ItemSmallName,ItemName,PriceMin,PriceMax,Gender,Age,FamilyCount,MemberYN,AgeGroup
0,U10001,U10001-O2023-1002,2023-01-06 17:08:51,L4-M17-S0530-1024,33310,2023-01-07 06:24:00,2023,1,6,17,...,S0530,전복,완도 활전복 1kg 중 22-25미,33160.0,37070.0,여성,26,2,Y,20
1,U10001,U10001-O2023-1002,2023-01-06 17:08:51,L1-M21-S0540-1082,3780,2023-01-07 06:24:00,2023,1,6,17,...,S0540,즉석,동원 양반 차돌된장찌개 (460G),3690.0,3970.0,여성,26,2,Y,20
2,U10001,U10001-O2023-1002,2023-01-06 17:08:51,L1-M15-S0140-1311,22520,2023-01-07 06:24:00,2023,1,6,17,...,S0140,냉동,오뚜기 듬뿍 새우볶음밥450g (2인분) x 5봉지 /,22150.0,23150.0,여성,26,2,Y,20
3,U10001,U10001-O2023-1002,2023-01-06 17:08:51,L4-M12-S0350-1035,21630,2023-01-07 06:24:00,2023,1,6,17,...,S0350,사과,[산지직송] 새콤달콤 부사 사과 5kg (13과내),20810.0,23030.0,여성,26,2,Y,20
4,U10001,U10001-O2023-1003,2023-01-13 16:50:14,L4-M12-S0640-1057,11700,2023-01-14 06:28:00,2023,1,13,16,...,S0640,토마토,스테비아 방울 토마토 라루 토망고 1kg,11640.0,13020.0,여성,26,2,Y,20


In [11]:
df_merged.columns

Index(['idUser', 'idOrder', 'OrderDT', 'ItemCode', 'Price', 'DeliveryDT',
       'OrderYear', 'OrderMonth', 'OrderDay', 'OrderHour', 'OrderMinute',
       'OrderWeekday', 'DeliveryYear', 'DeliveryMonth', 'DeliveryDay',
       'DeliveryHour', 'DeliveryMinute', 'DeliveryWeekday', 'DeliveryDeadline',
       'IsLate', 'ItemLargeCode', 'ItemLargeName', 'ItemMiddleCode',
       'ItemMiddleName', 'ItemSmallCode', 'ItemSmallName', 'ItemName',
       'PriceMin', 'PriceMax', 'Gender', 'Age', 'FamilyCount', 'MemberYN',
       'AgeGroup'],
      dtype='object')

In [12]:
df_merged.isnull().sum()

idUser                0
idOrder               0
OrderDT               0
ItemCode              0
Price                 0
DeliveryDT            0
OrderYear             0
OrderMonth            0
OrderDay              0
OrderHour             0
OrderMinute           0
OrderWeekday          0
DeliveryYear          0
DeliveryMonth         0
DeliveryDay           0
DeliveryHour          0
DeliveryMinute        0
DeliveryWeekday       0
DeliveryDeadline      0
IsLate                0
ItemLargeCode       437
ItemLargeName       437
ItemMiddleCode      437
ItemMiddleName      437
ItemSmallCode       437
ItemSmallName       437
ItemName            437
PriceMin            437
PriceMax            437
Gender                0
Age                   0
FamilyCount           0
MemberYN              0
AgeGroup              0
dtype: int64

In [13]:
missing_items = df_merged[df_merged['ItemName'].isnull()][['ItemCode', 'OrderYear']]                                                                                              
print(missing_items.value_counts()) 

ItemCode           OrderYear
L4-M22-S0650-1057  2023         126
L4-M17-S0030-1002  2024           4
L4-M22-S0020-1064  2024           4
L4-M22-S0650-1008  2024           3
L4-M22-S0360-1039  2024           3
                               ... 
L1-M17-S0260-1068  2024           1
L1-M17-S0260-1079  2024           1
L1-M17-S0260-1091  2023           1
L1-M17-S0260-1102  2024           1
L5-M25-S0630-1021  2024           1
Name: count, Length: 260, dtype: int64
